# 01 — Setup, Annotations, and Video Acquisition

Prepares the AVUT-Human dataset.

**What it does**
1. Mounts Drive, clones repo, installs deps.
2. Downloads `AV_Human_data.json` from `tsinghua-ee/AVUTBenchmark`.
3. Picks a balanced sample (configurable `n_per_task`).
4. Downloads videos directly from the HF dataset (NOT yt-dlp — the AVUT
   authors host the mp4s themselves, so it's much more reliable).

**Runtime:** ~15-30 min on Colab depending on sample size and bandwidth.
**GPU:** not needed for this notebook; CPU runtime is fine.

Resume-safe: rerun to skip already-downloaded files.


In [ ]:
# ─── Colab bootstrap ────────────────────────────────────────
# Mount Drive (caches model weights + videos across sessions),
# clone the repo, install dependencies, and configure the cache dirs.
import os, sys, subprocess, pathlib

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/avut"
    REPO_DIR = "/content/omnimodel-research"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if not os.path.exists(REPO_DIR):
        subprocess.run([
            "git", "clone",
            "https://github.com/samadasyed/omnimodel-research.git",
            REPO_DIR,
        ], check=True)
else:
    DRIVE_ROOT = os.path.expanduser("~/avut")
    REPO_DIR = str(pathlib.Path.cwd())
    os.makedirs(DRIVE_ROOT, exist_ok=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Persistent storage layout (data + model caches under DRIVE_ROOT)
DATA_DIR        = os.path.join(DRIVE_ROOT, "data")
VIDEO_DIR       = os.path.join(DATA_DIR, "videos")
AUDIO_DIR       = os.path.join(DATA_DIR, "audio")
SILENT_DIR      = os.path.join(DATA_DIR, "silent")
TRANSCRIPT_DIR  = os.path.join(DATA_DIR, "transcripts")
ANNOTATION_DIR  = os.path.join(DATA_DIR, "annotations")
RESULTS_DIR     = os.path.join(DRIVE_ROOT, "results")
RAW_PRED_DIR    = os.path.join(RESULTS_DIR, "raw_predictions")
METRICS_DIR     = os.path.join(RESULTS_DIR, "metrics")
HF_CACHE        = os.path.join(DRIVE_ROOT, ".cache", "hf")
WHISPER_CACHE   = os.path.join(DRIVE_ROOT, ".cache", "whisper")

for d in [VIDEO_DIR, AUDIO_DIR, SILENT_DIR, TRANSCRIPT_DIR,
          ANNOTATION_DIR, RAW_PRED_DIR, METRICS_DIR, HF_CACHE, WHISPER_CACHE]:
    os.makedirs(d, exist_ok=True)

# Redirect HF cache to Drive so we don't redownload weights each session
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_CACHE

print(f"Repo:       {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"HF cache:   {HF_CACHE}")


In [ ]:
# ─── Install model dependencies ──────────────────────────
# transformers (Gemma-3n preview support landed in 4.51), accelerate,
# soundfile for audio I/O, openai-whisper for ASR. cv2 (opencv-python)
# is preinstalled on Colab and used for frame sampling.
%pip install -q -U "transformers>=4.51.0" "accelerate>=0.30"     "huggingface-hub>=0.24" "soundfile>=0.12" scipy
%pip install -q openai-whisper
# Qwen-Omni preview branch is only needed for notebook 05; install there.

# Gemma-3n is GATED on HuggingFace — you must accept the license at
# https://huggingface.co/google/gemma-3n-E2B-it and authenticate.
# In Colab: Tools → Secrets → add HF_TOKEN, then:
import os
if "HF_TOKEN" in os.environ or "HUGGINGFACE_TOKEN" in os.environ:
    from huggingface_hub import login
    login(token=os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN"))
    print("HF authenticated")
else:
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            from huggingface_hub import login
            login(token=tok)
            print("HF authenticated (from Colab Secrets)")
        else:
            print("WARN: no HF_TOKEN — Gemma-3n download will fail. "
                  "Add HF_TOKEN to Colab Secrets first.")
    except Exception:
        print("WARN: not in Colab and no HF_TOKEN env var. "
              "Run `huggingface-cli login` before loading Gemma.")


## Download annotations

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

annotation_dst = os.path.join(ANNOTATION_DIR, "AV_Human_data.json")
if not os.path.exists(annotation_dst):
    src = hf_hub_download(
        repo_id="tsinghua-ee/AVUTBenchmark",
        filename="AV_Human_data.json",
        repo_type="dataset",
    )
    shutil.copy(src, annotation_dst)
    print(f"Saved {annotation_dst}")
else:
    print(f"Cached: {annotation_dst}")


## Inspect schema

In [ ]:
from src import data_utils

entries = data_utils.load_annotations(annotation_dst)
print(f"Loaded {len(entries)} valid entries")
print()
print("Task distribution (full set):")
for code, n in sorted(data_utils.task_distribution_by_code(entries).items()):
    print(f"  {code:5s}  {n:5d}")
print()
print("Sample entry:", entries[0])


## Pick a balanced sample

Set `N_PER_TASK` to control sample size. 20 is a quick pilot; 100 is a good Colab default; 280+ matches Jeff's full run.

In [ ]:
import json

N_PER_TASK = 100   # change me — 20 pilot, 100 default, 280+ full

samples = data_utils.balanced_subsample(
    entries, n_per_task=N_PER_TASK, seed=42,
)
print(f"Picked {len(samples)} samples ({N_PER_TASK}/task target)")
print()
for code, n in sorted(data_utils.task_distribution_by_code(samples).items()):
    print(f"  {code:5s}  {n}")

manifest_path = os.path.join(DATA_DIR, "sample_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(samples, f, indent=2)
print(f"\nManifest: {manifest_path}")


## Download videos from HuggingFace dataset

In [ ]:
counts = data_utils.download_videos_from_hf(samples, VIDEO_DIR)
print(counts)

# Filter manifest to only samples whose video downloaded successfully
samples_avail = data_utils.filter_to_available_videos(samples, VIDEO_DIR)
print(f"\nAvailable videos: {len(samples_avail)} / {len(samples)}")

# Save the filtered manifest — that's what subsequent notebooks read
manifest_avail_path = os.path.join(DATA_DIR, "sample_manifest_available.json")
with open(manifest_avail_path, "w") as f:
    json.dump(samples_avail, f, indent=2)
print(f"Available-only manifest: {manifest_avail_path}")


Next: run `02_preprocess.ipynb`.